<a href="https://colab.research.google.com/github/NikitaIvagin/ml-portfolio/blob/main/nlp/corporate_documentation_qa/Data_%D0%BF%D0%B0%D0%B9%D0%BF%D0%BB%D0%B0%D0%B9%D0%BD_%D0%B8_%D0%BF%D0%BE%D0%B8%D1%81%D0%BA%D0%BE%D0%B2%D0%B0%D1%8F_%D1%81%D0%B8%D1%81%D1%82%D0%B5%D0%BC%D0%B0_%D0%BF%D0%BE_%D0%BA%D0%BE%D1%80%D0%BF%D0%BE%D1%80%D0%B0%D1%82%D0%B8%D0%B2%D0%BD%D0%BE%D0%B9_%D0%BD%D0%BE%D1%80%D0%BC%D0%B0%D1%82%D0%B8%D0%B2%D0%BD%D0%BE%D0%B9_%D0%B4%D0%BE%D0%BA%D1%83%D0%BC%D0%B5%D0%BD%D1%82%D0%B0%D1%86%D0%B8%D0%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

В данном блокноте приводится код для создания нейро-сотрудника, который будет отвечать на вопросы сотрудников компании по ее документам. В качестве таких документов были выбраны единое положение о закупках и его приложения, взятые с сайта Ростеха. Поскольку это официальный документ крупной компании, считаем его достаточно структурированным для использования без изменений.

Фреймворк для реализации программы - LlamaIndex.

LLM - saiga mistral 7b. Для систем без gpu так же предусмотрена ячейка кода с импортом более легкой Qwen, но из-за того, что изначально она обучена под китайский язык, увеличена вероятность галлюцинаций вполь до замены русских слов на иероглифы.

В связи с ограничениями ресурсов в бесплатной версии Colab была выбрана векторная база для создания RAG-системы. Стоит отметить, что это снижает точность ответов нейросети, поэтому в нашем контексте (ответ по документам) такая реализация может служить только прототипом, для быстрой и дешевой демонстрации проекта заказчику.

Во время разработки нейросотрудника использовал opentelemetry для его проверки. Были обнаружены базовые галлюцинации: модель отвечала по контексту, которого не существует, либо отвечала не по тому документу, по которому подразумевался проверочный вопрос. Для "победы" над галлюцинациями были использованы следующие методы: улучшение промпта (модель скажет "Я не знаю" в случае отсутствия информации в документе), и добавления поиска по нескольким подходящим документам.

Для улучшения работы нейро-сотрудника были добавлены переранжирование и гибридный ретривер.

В качестве системы безопасности была сделана проверка запроса пользователя на токсичность, следовательно, такая проверка отсекает нежелательные моменты, такие как ответы оскорблением на оскорбление.

In [ ]:
%%writefile requirements.txt
llama-index
llama-index-llms-huggingface
transformers
accelerate
bitsandbytes
llama-index-readers-file
llama-index-embeddings-huggingface
docx2txt
triton
llama-index-retrievers-bm25

Writing requirements.txt


In [ ]:
%pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.1/70.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 683.3/683.3 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/5

In [ ]:
%pip install --upgrade llama-index-core llama-index-retrievers-bm25 llama-index

In [ ]:
from google.colab import drive
import triton
drive.mount('/content/drive')

from huggingface_hub import login
HF_TOKEN = getpass.getpass("Введите HF_TOKEN:")
login(HF_TOKEN, add_to_git_credential=True)

Mounted at /content/drive


In [ ]:
from llama_index.llms.huggingface import HuggingFaceLLM
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

# Имя модели (Saiga 7B на основе Mistral)
model_name = "IlyaGusev/saiga_mistral_7b"

# Конфигурация 4-битного квантования
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# Загрузка токенизатора и модели на GPU с квантованием
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",      # автоматически на GPU
    torch_dtype=torch.float16,
)

# Создание LLM для LlamaIndex
llm = HuggingFaceLLM(
    model=model,
    tokenizer=tokenizer,
    context_window=4096,
    max_new_tokens=512,
    generate_kwargs={
        "temperature": 0.3,
        "do_sample": True,
        "eos_token_id": tokenizer.eos_token_id,
        "pad_token_id": tokenizer.eos_token_id,   # чтобы избежать бесконечной генерации
        "repetition_penalty": 1.1,                # подавление повторов
    },
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

...

pytorch_model-00001-of-00002.bin:   0%|          | 0.00/9.94G [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/120 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/54.6M [00:00<?, ?B/s]

In [ ]:
Qwen = '''# Импорт модели
from llama_index.llms.huggingface import HuggingFaceLLM
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

llm = HuggingFaceLLM(
    model=model,
    tokenizer=tokenizer,
    context_window=32768,
    max_new_tokens=512,
    generate_kwargs={
        "temperature": 0.2,
        "do_sample": True,
        "eos_token_id": tokenizer.eos_token_id,
        "pad_token_id": tokenizer.eos_token_id,
        "repetition_penalty": 1.1,                   # снижает повторы
    },
)'''

In [ ]:
# Импорт документов
from llama_index.core import SimpleDirectoryReader

folder_path = "/content/drive/MyDrive/Ростех"
reader = SimpleDirectoryReader(
    input_dir=folder_path,
    required_exts=[".docx"],
    recursive=False           # не загружаем файлы из подпапок
)
documents = reader.load_data()

print(f"Successfully loaded {len(documents)} documents.")

Successfully loaded 16 documents.


In [ ]:
from llama_index.core import VectorStoreIndex
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.retrievers import BaseRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.response_synthesizers import get_response_synthesizer
from llama_index.core.postprocessor import SentenceTransformerRerank
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import NodeWithScore
from typing import List
from collections import defaultdict

# Создание векторного индекса
node_parser = SentenceSplitter(chunk_size=512, chunk_overlap=50)
embed_model = HuggingFaceEmbedding(model_name="cointegrated/rubert-tiny2")

vector_index = VectorStoreIndex.from_documents(
    documents,
    llm=llm,
    embed_model=embed_model,
    node_parser=node_parser
)
print("Векторная база данных создана.")

# Гибридный ретривет (BM25 + векторный)
VECTOR_TOP_K = 8
BM25_TOP_K = 8

vector_retriever = VectorIndexRetriever(index=vector_index, similarity_top_k=VECTOR_TOP_K)
bm25_retriever = BM25Retriever.from_defaults(
    docstore=vector_index.docstore,
    similarity_top_k=BM25_TOP_K
)

def fuse_results(vector_nodes: List[NodeWithScore],
                 bm25_nodes: List[NodeWithScore],
                 k: int = 60) -> List[NodeWithScore]:
    """Reciprocal Rank Fusion (RRF)"""
    scores = defaultdict(float)
    for rank, node in enumerate(vector_nodes, start=1):
        scores[node.node.node_id] += 1.0 / (k + rank)
    for rank, node in enumerate(bm25_nodes, start=1):
        scores[node.node.node_id] += 1.0 / (k + rank)
    sorted_ids = sorted(scores.keys(), key=lambda nid: scores[nid], reverse=True)
    combined = []
    for node_id in sorted_ids:
        orig = next((n for n in vector_nodes + bm25_nodes if n.node.node_id == node_id), None)
        if orig:
            combined.append(NodeWithScore(node=orig.node, score=scores[node_id]))
    return combined

class HybridRetriever(BaseRetriever):
    def __init__(self, vector_retriever, bm25_retriever):
        self.vector_retriever = vector_retriever
        self.bm25_retriever = bm25_retriever
        super().__init__()
    def _retrieve(self, query: str) -> List[NodeWithScore]:
        vector_nodes = self.vector_retriever.retrieve(query)
        bm25_nodes = self.bm25_retriever.retrieve(query)
        return fuse_results(vector_nodes, bm25_nodes)

hybrid_retriever = HybridRetriever(vector_retriever, bm25_retriever)

# Переранжирование
reranker = SentenceTransformerRerank(
    model="cross-encoder/ms-marco-MiniLM-L-2-v2",
    top_n=3
)

# Создание query engine
response_synthesizer = get_response_synthesizer(llm=llm)

query_engine = RetrieverQueryEngine(
    retriever=hybrid_retriever,
    response_synthesizer=response_synthesizer,
    node_postprocessors=[reranker],
)

print("Гибридный RAG с переранжированием готов.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/118M [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Гибридный RAG с переранжированием готов.


In [ ]:
# Настройка нейро-сотрудника
settings =  '''Ты эксперт по закупочной деятельности.
                        Твоя задача ответить так, чтобы у собеседника не осталось неясностей.
                        Тон общения: сухой, фактологический, ссылочный. Минимум воды.
                        Отвечай максимально точно по документу, не придумывай ничего от себя.
                        Отвечай по-русски.
                        Если запрашиваемой информации в документе нет (и только в этом случае!), ответь "Я не знаю".
                        Вот запрос от пользователя: " '''
end = ''' " '''

# Функция для запроса к нейро-сотруднику
def ask(message):
    response_vector = query_engine.query(settings + message + end)
    print(response_vector.response)

In [ ]:
# Создание фильтра для проверки запроса пользователя
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F

model_name = "cointegrated/rubert-tiny-toxicity"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

def classify_safety(text: str, threshold: float = 0.1) -> str:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        logits = model(**inputs).logits
        # Модель имеет 2 выхода: [non-toxic, toxic]
        probs = F.softmax(logits, dim=-1)
        # Класс 1 — токсичный
        toxic_score = probs[0][1].item()

    print(f"DEBUG: toxic score = {toxic_score:.4f} for text: {text}")  # отладка
    return "unsafe" if toxic_score > threshold else "safe"

# Тест
test_texts = [
    "Ты дурак, иди в жопу",
    "Ты пепе",
    "Привет, как дела?"
]

for t in test_texts:
    print(classify_safety(t, threshold=0.1))

tokenizer_config.json:   0%|          | 0.00/377 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/957 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/47.2M [00:00<?, ?B/s]

DEBUG: toxic score = 0.9700 for text: Ты дурак, иди в жопу
unsafe
DEBUG: toxic score = 0.0032 for text: Ты пепе
safe
DEBUG: toxic score = 0.0000 for text: Привет, как дела?
safe


In [ ]:
# Функция для запроса с фильтром
def ask_with_guard(message):
    guard_decision = classify_safety(message)

    print(f"Safety classification decision: {guard_decision}")

    if guard_decision == "safe":
        response_vector = query_engine.query(settings + message + end)
        print(response_vector.response)
    else:
        print("Ваш запрос был оценен как небезопасный.")

ask_with_guard("Ты дурак, иди в жопу")

DEBUG: toxic score = 0.9700 for text: Ты дурак, иди в жопу
Safety classification decision: unsafe
Ваш запрос был оценен как небезопасный.


In [ ]:
# Вопрос к нейро-сотруднику по документам
ask_with_guard("Чем будет результат квалификационного отбора для серии закупок?")

DEBUG: toxic score = 0.0000 for text: Чем будет результат квалификационного отбора для серии закупок?
Safety classification decision: safe

"В результате квалификационного отбора для серии закупок будут определены участники, которые успешно прошли все этапы отбора и имеют право на участие в будущих закупках."
